In [1]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import pandas as pd
import numpy as np
from tqdm import tqdm
from re import sub
import re
tqdm.pandas()

C:\Libraries\anaconda3\lib\site-packages\tqdm\std.py:697: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [41]:
word_vectors = Word2Vec.load("./sentiment140-word2vec_defaults.model").wv

In [71]:
word_vectors = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)

In [4]:
dataframe = pd.read_csv("./sentiment140-cleaned.csv", encoding = "ISO-8859-1", engine="python")

In [7]:
dataframe.head()

,sentiment,user,text
0,0,_TheSpecialOne_,@switchfoot awww that is a bummer you shoulda ...
1,0,scotthamilton,is upset that he can t update his facebook by ...
2,0,mattycus,@kenichan i dived many times for the ball mana...
3,0,ElleCTF,my whole body feels itchy and like its on fire
4,0,Karoli,@nationwideclass no it is not behaving at all ...


In [52]:
n=100000

In [53]:
df = dataframe.sample(n=n)

In [54]:
df['text'] = df.progress_apply(lambda row: row['text'].split(), axis=1)

100%|██████████| 100000/100000 [00:01<00:00, 76224.19it/s]


In [55]:
cluster_good = ["good", "nice", "cool", "lovely", "wonderful", "great", "awesome", "fantastic", "amazing", "fun", "excellent"]
cluster_bad = ["bad", "horrible", "terrible", "awful", "worst", "shitty", "crappy", "sucks", "hate"]

In [56]:
def clusterDistance(word, cluster):
    return np.average(word_vectors.distances(word, cluster))

def tweetClusterDistance(text, cluster):
    out = []
    for word in text:
        if not (word in word_vectors.vocab):
            continue
        out.append(clusterDistance(word, cluster))
    return np.average(out)

In [57]:
def predictTweet(text):
    wordvectors = []
    for word in text:
        if not (word in word_vectors.vocab):
            continue
        wordvectors.append(word_vectors[word])
    if len(wordvectors) == 0:
        return 0
    sentencevector = np.mean(wordvectors, axis=0)

    positive = clusterDistance(sentencevector, cluster_good)
    negative = clusterDistance(sentencevector, cluster_bad)
    if positive < negative:
        return 4
    return 0

df['positive'] = df.progress_apply(lambda row: tweetClusterDistance(row['text'], cluster_good), axis=1)

df['negative'] = df.progress_apply(lambda row: tweetClusterDistance(row['text'], cluster_bad), axis=1)

def getSentiment(row):
    if row['positive'] < row['negative']:
        return 4 #positive
    return 0 #negative

df['predict'] = df.progress_apply(lambda row: getSentiment(row), axis=1)

In [58]:
df['predict'] = df.progress_apply(lambda row: predictTweet(row['text']), axis=1)
df['predict_correct'] = df.progress_apply(lambda row: row['sentiment'] == row['predict'], axis=1)
print(df['predict_correct'].value_counts()[True] / n)

100%|██████████| 100000/100000 [00:01<00:00, 97633.75it/s]0.66684



In [59]:
print(df['predict_correct'].value_counts()[True] / n)

0.66684


In [60]:
wrong = df.loc[df['predict_correct'] == False]
wrong['sentiment'].value_counts()

0    23889
4     9427
Name: sentiment, dtype: int64

In [61]:
right = df.loc[df['predict_correct'] == True]
right['sentiment'].value_counts()


4    40646
0    26038
Name: sentiment, dtype: int64